# 08 · ReAct 与 Plan-and-Execute

- **ReAct：**真实 LLM 在 Thought → Action → Observation 循环中选择工具；无 tool call 即结束。
- **Plan-and-Execute：**真实 LLM Planner 生成结构化计划，Worker 执行，真实 LLM Evaluator 决定结束或有限次数 Replan。

两条链路都必须使用 `.env` 配置的真实模型；缺少配置时立即失败，不提供离线降级。

In [ ]:
from __future__ import annotations

from pathlib import Path
import os

from dotenv import load_dotenv


def find_repo_root() -> Path:
    '''向上找到含 pyproject.toml 的仓库根，避免 notebook 工作目录不在根上。'''
    here = Path.cwd()
    for p in [here, *here.parents]:
        if (p / "pyproject.toml").exists():
            return p
    return here


ROOT = find_repo_root()
os.chdir(ROOT)
load_dotenv(ROOT / ".env")
missing_llm = [
    name for name in ("LLM_BASE_URL", "LLM_MODEL")
    if not (os.getenv(name) or "").strip()
]
if missing_llm:
    raise ValueError(
        "真实 LLM ReAct/Plan-and-Execute 必须配置 .env，缺少："
        + ", ".join(missing_llm)
    )
print("cwd =", ROOT)
print("mode = live LLM required")

: 

In [ ]:
def show_graph(graph):
    """在 notebook 展示图结构 PNG（课表：看得见 State / Node / Edge）。"""
    from IPython.display import Image, display

    try:
        png = graph.get_graph().draw_mermaid_png()
        display(Image(png))
    except Exception as e:
        print("PNG 不可用，打印 mermaid：", e)
        print(graph.get_graph().draw_mermaid())


In [ ]:
from typing import Annotated, Literal, TypedDict

from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode

from hello_agent.config import get_llm


@tool
def search_rules(query: str) -> str:
    '''检索订舱/危险品规则摘要。'''
    if "锂" in query:
        return "锂电池：MSDS + 危申；核对两港限制。"
    return "请补充港口与货种。"


class ReActState(TypedDict):
    messages: Annotated[list, add_messages]


base_llm = get_llm(temperature=0)
react_llm = base_llm.bind_tools([search_rules])


def decide(state: ReActState) -> dict:
    response = react_llm.invoke([
        SystemMessage(content=(
            "你是订舱规则 Agent。回答规则问题前必须调用 search_rules；"
            "收到 ToolMessage 后基于工具证据给出简洁终答，不得再次调用工具。"
        )),
        *state["messages"],
    ])
    return {"messages": [response]}


def route(state: ReActState) -> Literal["tools", "end"]:
    return "tools" if getattr(state["messages"][-1], "tool_calls", None) else "end"


builder = StateGraph(ReActState)
builder.add_node("agent", decide)
builder.add_node("tools", ToolNode([search_rules]))
builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", route, {"tools": "tools", "end": END})
builder.add_edge("tools", "agent")
agent = builder.compile()

show_graph(agent)
print("mode = live LLM ReAct trace")

In [ ]:
result = agent.invoke(
    {"messages": [{"role": "user", "content": "上海→洛杉矶 锂电池订舱要什么文件？"}]},
    {"recursion_limit": 6},
)
for message in result["messages"]:
    print(type(message).__name__, ":", getattr(message, "content", "")[:200])

tool_messages = [message for message in result["messages"] if isinstance(message, ToolMessage)]
final_answer = result["messages"][-1]
assert tool_messages, "真实模型未产生 ToolMessage"
assert not getattr(final_answer, "tool_calls", None)
assert str(getattr(final_answer, "content", "")).strip()
print("PASS: live LLM ReAct contains ToolMessage and final answer")

## Plan-and-Execute 结构

Planner 产出任务列表 → `Send` 并行 Worker → Evaluator 审核 → 可选 Replan（本例最多一次）→ Finalizer。下一节给出完整真实 LLM 实现。

In [ ]:
print("Goal → LLM Planner → Send Workers → LLM Evaluator → Result")
print("           ↑________ Command Replan (max 1) ________↓")

<!-- codex:p0:08 -->
## P0 进阶 · 可运行的 Plan-and-Execute

下面把前面的结构示意升级成一张由真实 LLM 驱动的图：

1. 真实 LLM Planner 通过结构化输出生成本轮任务；
2. 条件边返回多个 `Send`，动态并行创建 Worker；
3. reducer 汇总 Worker 证据；
4. 真实 LLM Evaluator 审核证据，并返回 `Command(update=..., goto=...)` 选择 Replan 或结束；
5. `replan_count` 与 `recursion_limit` 共同限制循环。

In [ ]:
import operator

from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from langgraph.types import Command, Send


TaskName = Literal["equipment", "route", "dangerous_goods"]


class BookingPlan(BaseModel):
    tasks: list[TaskName] = Field(description="本轮需要执行的审核任务")


class EvidenceEvaluation(BaseModel):
    approved: bool = Field(description="证据是否覆盖全部三个审核维度")
    feedback: str = Field(description="对证据完整性的简短说明")


planner_parser = PydanticOutputParser(pydantic_object=BookingPlan)
evaluator_parser = PydanticOutputParser(pydantic_object=EvidenceEvaluation)


class PlanState(TypedDict, total=False):
    booking_id: str
    tasks: list[str]
    evidence: Annotated[list[str], operator.add]
    replan_count: int
    evaluation: str
    result: str


class WorkerState(TypedDict):
    task: str


def planner(state: PlanState) -> dict:
    replan_count = state.get("replan_count", 0)
    if replan_count == 0:
        instruction = (
            "这是首轮教学计划。必须且只能选择 equipment 和 route，"
            "故意暂不选择 dangerous_goods，以便下一轮演示 Replan。"
        )
    else:
        instruction = (
            "这是唯一一次 Replan。已有 evidence 如下："
            f"{state.get('evidence', [])}。必须且只能选择尚缺的 dangerous_goods。"
        )
    plan_message = base_llm.invoke(
        [
            SystemMessage(content=(
                "你是订舱预审 Planner。严格按照用户给出的本轮约束输出 JSON；"
                "不得添加枚举外任务，不得输出解释。\n"
                + planner_parser.get_format_instructions()
            )),
            HumanMessage(content=f"booking_id={state['booking_id']}。{instruction}"),
        ]
    )
    plan = planner_parser.invoke(plan_message)
    tasks = list(dict.fromkeys(plan.tasks))
    if not tasks:
        raise ValueError("真实 LLM Planner 未返回任何任务")
    print("LLM plan =", tasks)
    return {"tasks": tasks}


def dispatch_workers(state: PlanState):
    return [Send("worker", {"task": task}) for task in state["tasks"]]


def worker(state: WorkerState) -> dict:
    evidence_by_task = {
        "equipment": "equipment:ok",
        "route": "route:ok",
        "dangerous_goods": "dangerous_goods:manual-evidence-required",
    }
    return {"evidence": [evidence_by_task[state["task"]]]}


def evaluate(state: PlanState) -> Command[Literal["planner", "finalize"]]:
    evidence = state["evidence"]
    evaluation_message = base_llm.invoke(
        [
            SystemMessage(content=(
                "你是订舱证据 Evaluator。只有 evidence 同时包含 equipment、route、"
                "dangerous_goods 三类前缀时 approved 才能为 true。只输出 JSON，不得解释。\n"
                + evaluator_parser.get_format_instructions()
            )),
            HumanMessage(content=f"请审核 evidence={evidence}"),
        ]
    )
    evaluation = evaluator_parser.invoke(evaluation_message)
    prefixes = {item.split(":", 1)[0] for item in evidence}
    authoritative_complete = prefixes >= {"equipment", "route", "dangerous_goods"}
    if evaluation.approved != authoritative_complete:
        raise AssertionError(
            "LLM Evaluator 与明确证据规则不一致："
            f"approved={evaluation.approved}, evidence={evidence}"
        )
    if evaluation.approved:
        return Command(
            update={"evaluation": evaluation.feedback},
            goto="finalize",
        )
    if state.get("replan_count", 0) >= 1:
        raise RuntimeError("达到 max replan 后证据仍不完整")
    return Command(
        update={
            "replan_count": state.get("replan_count", 0) + 1,
            "evaluation": evaluation.feedback,
        },
        goto="planner",
    )


def finalize(state: PlanState) -> dict:
    return {"result": " | ".join(sorted(state["evidence"]))}


plan_builder = StateGraph(PlanState)
plan_builder.add_node("planner", planner)
plan_builder.add_node("worker", worker)
plan_builder.add_node("evaluate", evaluate)
plan_builder.add_node("finalize", finalize)
plan_builder.add_edge(START, "planner")
plan_builder.add_conditional_edges("planner", dispatch_workers)
plan_builder.add_edge("worker", "evaluate")
plan_builder.add_edge("finalize", END)
plan_execute_app = plan_builder.compile()

plan_result = plan_execute_app.invoke(
    {
        "booking_id": "BK-PLAN-001",
        "evidence": [],
        "replan_count": 0,
    },
    {"recursion_limit": 10},
)

print("evidence =", plan_result["evidence"])
print("replan_count =", plan_result["replan_count"])
print("evaluation =", plan_result["evaluation"])
print("result =", plan_result["result"])

assert plan_result["replan_count"] == 1
assert len(plan_result["evidence"]) == 3
assert "dangerous_goods:manual-evidence-required" in plan_result["evidence"]
print("08 Send + reducer + Command plan-execute ok")